In [9]:
!uv pip install pandas openpyxl tkinterdnd2

Resolved 9 packages in 179ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/481.45 KiB          
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/481.45 KiB        
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/481.45 KiB        
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/481.45 KiB        
⠙ Preparing packages... (0/1)------------------- 60.08 KiB/481.45 KiB        
⠙ Preparing packages... (0/1)------------------- 76.08 KiB/481.45 KiB        
⠙ Preparing packages... (0/1)------------------- 92.08 KiB/481.45 KiB        
⠙ Preparing packages... (0/1)------------------- 108.08 KiB/481.45 KiB       
⠙ Preparing packages... (0/1)------------------- 124.08 KiB/481.45 KiB       
⠙ Preparing packages... (0/1)------------------- 140.08 KiB/481.45 KiB       
⠙ Preparing packages... (0/1)------------------- 156.08 KiB/481.45 Ki

In [17]:


import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import sqlite3
import pandas as pd
import os
from tkinterdnd2 import DND_FILES, TkinterDnD

# Database file location
DB_FILE = "mtg_collection.db"

In [25]:
class SearchableMultiSelectDropdown(tk.Frame):
    def __init__(self, parent, options, label_text="Select Options"):
        super().__init__(parent)
        self.options = sorted(options)
        self.selected_items = set() # Stores selected strings
        self.label_text = label_text
        
        # Main Button
        self.menubutton = tk.Button(self, text=label_text, command=self.toggle_popup, relief="raised", width=15, anchor="w")
        self.menubutton.pack(fill="x")

        self.popup = None

    def toggle_popup(self):
        if self.popup and self.popup.winfo_exists():
            self.popup.destroy()
            return

        # Create Popup (Toplevel Window)
        self.popup = tk.Toplevel(self)
        self.popup.wm_overrideredirect(True) # Remove window borders
        self.popup.attributes('-topmost', True) # Keep on top
        
        # Calculate Position
        x = self.menubutton.winfo_rootx()
        y = self.menubutton.winfo_rooty() + self.menubutton.winfo_height()
        self.popup.geometry(f"250x300+{x}+{y}")

        # --- Search Bar ---
        search_frame = tk.Frame(self.popup, bg="#eeeeee", padx=5, pady=5)
        search_frame.pack(fill="x")
        self.search_var = tk.StringVar()
        self.search_var.trace("w", self.update_list)
        tk.Entry(search_frame, textvariable=self.search_var).pack(fill="x")

        # --- Listbox with Scrollbar ---
        list_frame = tk.Frame(self.popup)
        list_frame.pack(fill="both", expand=True)
        
        scrollbar = tk.Scrollbar(list_frame, orient="vertical")
        self.listbox = tk.Listbox(list_frame, selectmode="multiple", exportselection=False, yscrollcommand=scrollbar.set, height=10)
        scrollbar.config(command=self.listbox.yview)
        
        scrollbar.pack(side="right", fill="y")
        self.listbox.pack(side="left", fill="both", expand=True)
        
        # Bind click/select events
        self.listbox.bind("<<ListboxSelect>>", self.on_select)
        
        # --- Done Button ---
        btn = tk.Button(self.popup, text="Done / Close", command=self.close_popup, bg="#dddddd")
        btn.pack(fill="x")

        self.update_list() # Populate initial list

    def update_list(self, *args):
        search_term = self.search_var.get().lower()
        self.listbox.delete(0, "end")
        
        # Filter options based on search
        filtered = [opt for opt in self.options if search_term in str(opt).lower()]
        
        for i, opt in enumerate(filtered):
            self.listbox.insert("end", opt)
            # Re-select if it was previously selected
            if opt in self.selected_items:
                self.listbox.selection_set(i)

    def on_select(self, event):
        # Sync the UI selection with our set of selected_items
        # We check only visible items (due to filter)
        all_visible = self.listbox.get(0, "end")
        current_ui_selection_indices = self.listbox.curselection()
        
        for i, item in enumerate(all_visible):
            if i in current_ui_selection_indices:
                self.selected_items.add(item)
            else:
                if item in self.selected_items:
                    self.selected_items.remove(item)
        
        self.update_label()

    def close_popup(self):
        if self.popup:
            self.popup.destroy()

    def update_label(self):
        if not self.selected_items:
            self.menubutton.config(text=self.label_text)
        elif len(self.selected_items) == 1:
            self.menubutton.config(text=list(self.selected_items)[0])
        else:
            self.menubutton.config(text=f"{len(self.selected_items)} Selected")
            
    def get_selected(self):
        return list(self.selected_items)
        
    def update_options(self, new_options):
        self.options = sorted(list(set(new_options))) # Unique and sorted
        # Remove selected items that no longer exist in options
        self.selected_items = {x for x in self.selected_items if x in self.options}
        self.update_label()
        
    def clear_selection(self):
        self.selected_items.clear()
        self.update_label()

In [26]:
class MTGCardManager:
    def __init__(self, root):
        self.root = root
        self.root.title("MTG Card Sorter & Inventory")
        self.root.geometry("1400x850")

        self.root.drop_target_register(DND_FILES)
        self.root.dnd_bind('<<Drop>>', self.on_file_drop)

        # Style Configuration
        style = ttk.Style(self.root)
        style.theme_use("clam")
        style.configure("Treeview", rowheight=30, font=('Arial', 10)) 
        style.configure("Treeview.Heading", font=('Arial', 10, 'bold'))

        self.conn = sqlite3.connect(DB_FILE)
        self.create_table()

        # --- Data Variables ---
        self.var_name = tk.StringVar()
        self.var_color = tk.StringVar()
        self.var_quantity = tk.IntVar(value=1)
        self.var_type = tk.StringVar()
        self.var_mana_cost = tk.StringVar()
        self.var_pt = tk.StringVar()
        self.var_subtypes = tk.StringVar()
        self.var_keywords = tk.StringVar()
        self.var_ability = tk.StringVar()
        self.var_loyalty = tk.StringVar()

        self.editing_card_id = None 

        self.card_types = [
            "Land", "Legendary Creature", "Creature", "Instant", "Sorcery",
            "Legendary Enchantment", "Enchantment", "Planeswalker",
            "Legendary Artifact", "Artifact", "Artifact Equipment",
            "Artifact Vehicle", "Artifact Creature", "Artifact Treasure"
        ]
        
        self.colors = ["White", "Blue", "Black", "Red", "Green", "Colorless", "Multicolor"]
        
        self.tree_dict = {} 
        self.setup_ui()
        
        # Load Dropdown Options
        self.refresh_dropdowns()

    def create_table(self):
        cursor = self.conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS cards (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                color TEXT,
                quantity INTEGER,
                card_category TEXT,
                mana_cost TEXT,
                power_toughness TEXT,
                subtypes TEXT,
                keywords TEXT,
                ability TEXT,
                loyalty INTEGER,
                UNIQUE(name, card_category)
            )
        """)
        self.conn.commit()

    def get_distinct_values(self, column):
        """Standard fetch for single value columns."""
        cursor = self.conn.cursor()
        try:
            cursor.execute(f"SELECT DISTINCT {column} FROM cards WHERE {column} IS NOT NULL AND {column} != ''")
            return [row[0] for row in cursor.fetchall()]
        except:
            return []

    def get_split_subtypes(self):
        """Fetches subtypes and splits 'Cat, Warrior' into individual items."""
        cursor = self.conn.cursor()
        cursor.execute("SELECT DISTINCT subtypes FROM cards WHERE subtypes IS NOT NULL AND subtypes != ''")
        raw = cursor.fetchall()
        
        unique_subs = set()
        for (s_str,) in raw:
            # Split by comma and strip whitespace
            parts = [p.strip() for p in str(s_str).split(',')]
            unique_subs.update(parts)
        
        return sorted(list(unique_subs))

    def refresh_dropdowns(self):
        self.db_costs = self.get_distinct_values("mana_cost")
        self.db_subtypes = self.get_split_subtypes() # Use the split logic here
        
        if hasattr(self, 'filter_cost'):
            self.filter_cost.update_options(self.db_costs)
            self.filter_sub.update_options(self.db_subtypes)

    def setup_ui(self):
        self.create_input_frame()

        self.main_notebook = ttk.Notebook(self.root)
        self.main_notebook.pack(fill="both", expand=True, padx=5, pady=5)

        master_frame = tk.Frame(self.main_notebook)
        self.main_notebook.add(master_frame, text="  MASTER LIST  ")
        self.create_master_tab_content(master_frame)

        for color in self.colors:
            color_frame = tk.Frame(self.main_notebook)
            self.main_notebook.add(color_frame, text=f"  {color}  ")
            self.create_color_tab_content(color_frame, color)

        self.load_all_cards()

    def create_input_frame(self):
        input_frame = tk.LabelFrame(self.root, text="Card Details (Drag & Drop Excel Files Here)", padx=10, pady=5)
        input_frame.pack(fill="x", padx=10, pady=5)

        tk.Label(input_frame, text="Name:").grid(row=0, column=0, sticky="e")
        tk.Entry(input_frame, textvariable=self.var_name, width=25).grid(row=0, column=1, sticky="w")
        
        tk.Label(input_frame, text="Color:").grid(row=0, column=2, sticky="e")
        tk.Entry(input_frame, textvariable=self.var_color, width=15).grid(row=0, column=3, sticky="w")
        
        tk.Label(input_frame, text="Quantity:").grid(row=0, column=4, sticky="e")
        tk.Entry(input_frame, textvariable=self.var_quantity, width=5).grid(row=0, column=5, sticky="w")

        tk.Label(input_frame, text="Category:").grid(row=0, column=6, sticky="e")
        self.type_combo = ttk.Combobox(input_frame, textvariable=self.var_type, values=self.card_types, state="readonly", width=20)
        self.type_combo.grid(row=0, column=7, sticky="w")
        self.type_combo.bind("<<ComboboxSelected>>", self.update_fields_visibility)

        self.dynamic_frame = tk.Frame(input_frame)
        self.dynamic_frame.grid(row=1, column=0, columnspan=8, pady=5, sticky="w")
        
        self.lbl_mana = tk.Label(self.dynamic_frame, text="Mana Cost:")
        self.ent_mana = tk.Entry(self.dynamic_frame, textvariable=self.var_mana_cost, width=10)
        self.lbl_pt = tk.Label(self.dynamic_frame, text="P/T:")
        self.ent_pt = tk.Entry(self.dynamic_frame, textvariable=self.var_pt, width=8)
        self.lbl_sub = tk.Label(self.dynamic_frame, text="Subtypes:")
        self.ent_sub = tk.Entry(self.dynamic_frame, textvariable=self.var_subtypes, width=15)
        self.lbl_key = tk.Label(self.dynamic_frame, text="Keywords:")
        self.ent_key = tk.Entry(self.dynamic_frame, textvariable=self.var_keywords, width=15)
        self.lbl_ab = tk.Label(self.dynamic_frame, text="Ability:")
        self.ent_ab = tk.Entry(self.dynamic_frame, textvariable=self.var_ability, width=30)
        self.lbl_loy = tk.Label(self.dynamic_frame, text="Loyalty:")
        self.ent_loy = tk.Entry(self.dynamic_frame, textvariable=self.var_loyalty, width=5)

        btn_frame = tk.Frame(input_frame)
        btn_frame.grid(row=2, column=0, columnspan=8, pady=5)
        
        self.btn_submit = tk.Button(btn_frame, text="Add Card", command=self.add_card, bg="#dddddd", width=15)
        self.btn_submit.pack(side="left", padx=5)
        
        self.btn_delete = tk.Button(btn_frame, text="Delete Card", command=self.delete_card, bg="#ff9999", width=15, state="disabled")
        self.btn_delete.pack(side="left", padx=5)
        
        tk.Button(btn_frame, text="Clear Form", command=self.clear_form, bg="#dddddd", width=15).pack(side="left", padx=5)

    def create_master_tab_content(self, parent):
        search_frame = tk.LabelFrame(parent, text="Filter Master List", padx=5, pady=5)
        search_frame.pack(fill="x", padx=5, pady=5)

        # --- Use New Searchable Dropdowns ---
        tk.Label(search_frame, text="Colors:").pack(side="left", padx=2)
        self.filter_color = SearchableMultiSelectDropdown(search_frame, self.colors, "Select Colors")
        self.filter_color.pack(side="left", padx=5)

        tk.Label(search_frame, text="Categories:").pack(side="left", padx=2)
        self.filter_cat = SearchableMultiSelectDropdown(search_frame, self.card_types, "Select Types")
        self.filter_cat.pack(side="left", padx=5)

        tk.Label(search_frame, text="Costs:").pack(side="left", padx=2)
        self.filter_cost = SearchableMultiSelectDropdown(search_frame, [], "Select Cost") # Empty init
        self.filter_cost.pack(side="left", padx=5)
        
        tk.Label(search_frame, text="Subtypes:").pack(side="left", padx=2)
        self.filter_sub = SearchableMultiSelectDropdown(search_frame, [], "Select Subtype") # Empty init
        self.filter_sub.pack(side="left", padx=5)

        tk.Button(search_frame, text="Search", command=self.search_master, bg="#b3d9ff", width=10).pack(side="left", padx=15)
        tk.Button(search_frame, text="Reset", command=self.clear_search, bg="#dddddd").pack(side="left")

        tree_frame = tk.Frame(parent)
        tree_frame.pack(fill="both", expand=True, padx=5, pady=5)
        
        if "Master" not in self.tree_dict: self.tree_dict["Master"] = {}
        self.tree_dict["Master"]["All"] = self.create_treeview(tree_frame)

    def search_master(self):
        self.tree_dict["Master"]["All"].delete(*self.tree_dict["Master"]["All"].get_children())

        sel_colors = self.filter_color.get_selected()
        sel_cats = self.filter_cat.get_selected()
        sel_costs = self.filter_cost.get_selected()
        sel_subs = self.filter_sub.get_selected()

        query = "SELECT name, color, quantity, card_category, mana_cost, power_toughness, subtypes, keywords, ability, loyalty FROM cards WHERE 1=1"
        params = []

        if sel_colors:
            clauses = []
            for c in sel_colors:
                clauses.append("color LIKE ?")
                params.append(f"%{c}%")
            query += " AND (" + " OR ".join(clauses) + ")"

        if sel_cats:
            placeholders = ",".join("?" * len(sel_cats))
            query += f" AND card_category IN ({placeholders})"
            params.extend(sel_cats)

        if sel_costs:
            clauses = []
            for c in sel_costs:
                clauses.append("mana_cost = ?")
                params.append(c)
            query += " AND (" + " OR ".join(clauses) + ")"

        if sel_subs:
            # Here we match partial strings. If "Cat" is selected, "Cat, Warrior" matches.
            clauses = []
            for s in sel_subs:
                clauses.append("subtypes LIKE ?")
                params.append(f"%{s}%")
            query += " AND (" + " OR ".join(clauses) + ")"

        query += " ORDER BY name ASC"

        cursor = self.conn.cursor()
        cursor.execute(query, params)
        for row in cursor.fetchall():
            self.tree_dict["Master"]["All"].insert("", "end", values=self.clean_row(row))

    def clear_search(self):
        self.filter_color.clear_selection()
        self.filter_cat.clear_selection()
        self.filter_cost.clear_selection()
        self.filter_sub.clear_selection()
        self.load_all_cards()

    def on_file_drop(self, event):
        file_path = event.data
        if file_path.startswith('{') and file_path.endswith('}'):
            file_path = file_path[1:-1]
        
        if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
            self.process_excel_file(file_path)
            self.conn.commit()
            self.refresh_dropdowns()
            self.load_all_cards()
            messagebox.showinfo("Import Successful", f"Imported: {os.path.basename(file_path)}")
        else:
            messagebox.showerror("Error", "Please drop a valid Excel file (.xlsx, .xls)")

    def create_color_tab_content(self, parent, color_name):
        inner_notebook = ttk.Notebook(parent)
        inner_notebook.pack(fill="both", expand=True, padx=5, pady=5)
        
        if color_name not in self.tree_dict: self.tree_dict[color_name] = {}

        all_frame = tk.Frame(inner_notebook)
        inner_notebook.add(all_frame, text=f" All {color_name} ")
        self.tree_dict[color_name]["All"] = self.create_treeview(all_frame)

        for ctype in self.card_types:
            type_frame = tk.Frame(inner_notebook)
            inner_notebook.add(type_frame, text=f" {ctype} ")
            self.tree_dict[color_name][ctype] = self.create_treeview(type_frame)

    def create_treeview(self, parent_frame):
        columns = ("Qty", "Name", "Color", "Category", "Cost", "P/T", "Subtypes", "Keywords", "Ability", "Loyalty")
        tree = ttk.Treeview(parent_frame, columns=columns, show="headings")
        
        col_widths = [40, 250, 80, 120, 60, 50, 180, 120, 300, 50]
        for col, width in zip(columns, col_widths):
            tree.heading(col, text=col)
            tree.column(col, width=width, minwidth=50)

        v_scroll = ttk.Scrollbar(parent_frame, orient=tk.VERTICAL, command=tree.yview)
        tree.configure(yscroll=v_scroll.set)
        v_scroll.pack(side="right", fill="y")

        h_scroll = ttk.Scrollbar(parent_frame, orient=tk.HORIZONTAL, command=tree.xview)
        tree.configure(xscroll=h_scroll.set)
        h_scroll.pack(side="bottom", fill="x")

        tree.pack(side="left", fill="both", expand=True)
        tree.bind("<Double-1>", self.on_tree_double_click)
        return tree

    def on_tree_double_click(self, event):
        tree = event.widget
        selection = tree.selection()
        if not selection: return
        item = tree.item(selection[0])
        values = item['values']
        name = values[1]
        category = values[3]
        
        cursor = self.conn.cursor()
        cursor.execute("SELECT * FROM cards WHERE name=? AND card_category=?", (name, category))
        row = cursor.fetchone()
        
        if row:
            self.editing_card_id = row[0]
            self.var_name.set(row[1])
            self.var_color.set(row[2] if row[2] else "")
            self.var_quantity.set(row[3])
            self.var_type.set(row[4])
            self.var_mana_cost.set(row[5] if row[5] else "")
            self.var_pt.set(row[6] if row[6] else "")
            self.var_subtypes.set(row[7] if row[7] else "")
            self.var_keywords.set(row[8] if row[8] else "")
            self.var_ability.set(row[9] if row[9] else "")
            self.var_loyalty.set(row[10] if row[10] else "")
            
            self.update_fields_visibility()
            self.btn_submit.config(text="Save Changes", bg="#b3d9ff", command=self.save_edits)
            self.btn_delete.config(state="normal")

    def save_edits(self):
        if self.editing_card_id is None: return
        try:
            name = self.var_name.get().strip()
            cat = self.var_type.get()
            if not name or not cat:
                messagebox.showerror("Error", "Name/Category required.")
                return

            cursor = self.conn.cursor()
            query = """UPDATE cards SET name=?, color=?, quantity=?, card_category=?, mana_cost=?, power_toughness=?, subtypes=?, keywords=?, ability=?, loyalty=? WHERE id=?"""
            data = (name, self.var_color.get(), self.var_quantity.get(), cat, self.var_mana_cost.get(), self.var_pt.get(), self.var_subtypes.get(), self.var_keywords.get(), self.var_ability.get(), self.var_loyalty.get() if self.var_loyalty.get() else None, self.editing_card_id)
            cursor.execute(query, data)
            self.conn.commit()
            
            self.refresh_dropdowns()
            messagebox.showinfo("Success", "Updated.")
            self.load_all_cards()
            self.clear_form()
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def delete_card(self):
        if self.editing_card_id is None: return
        if messagebox.askyesno("Confirm", f"Delete '{self.var_name.get()}'?"):
            self.conn.execute("DELETE FROM cards WHERE id=?", (self.editing_card_id,))
            self.conn.commit()
            self.refresh_dropdowns()
            self.load_all_cards()
            self.clear_form()

    def add_card(self):
        name = self.var_name.get().strip()
        cat = self.var_type.get()
        if not name or not cat:
            messagebox.showerror("Error", "Name/Category required.")
            return

        data = {
            "name": name, "color": self.var_color.get(), "quantity": self.var_quantity.get(),
            "card_category": cat, "mana_cost": self.var_mana_cost.get(), "power_toughness": self.var_pt.get(),
            "subtypes": self.var_subtypes.get(), "keywords": self.var_keywords.get(),
            "ability": self.var_ability.get(), "loyalty": self.var_loyalty.get() if self.var_loyalty.get() else None
        }
        
        self.upsert_card(data)
        self.conn.commit()
        self.refresh_dropdowns()
        self.load_all_cards()
        self.clear_form()

    def upsert_card(self, data):
        cursor = self.conn.cursor()
        cursor.execute("SELECT id, quantity FROM cards WHERE name=? AND card_category=?", (data['name'], data['card_category']))
        result = cursor.fetchone()
        if result:
            cursor.execute("UPDATE cards SET quantity=? WHERE id=?", (result[1] + data['quantity'], result[0]))
        else:
            cursor.execute("""INSERT INTO cards (name, color, quantity, card_category, mana_cost, power_toughness, subtypes, keywords, ability, loyalty) VALUES (:name, :color, :quantity, :card_category, :mana_cost, :power_toughness, :subtypes, :keywords, :ability, :loyalty)""", data)

    def load_all_cards(self):
        for main_cat in self.tree_dict:
            for sub_cat in self.tree_dict[main_cat]:
                self.tree_dict[main_cat][sub_cat].delete(*self.tree_dict[main_cat][sub_cat].get_children())
        
        cursor = self.conn.cursor()
        cursor.execute("SELECT name, color, quantity, card_category, mana_cost, power_toughness, subtypes, keywords, ability, loyalty FROM cards ORDER BY name ASC")
        for row in cursor.fetchall():
            self.insert_row_to_ui(row)

    def insert_row_to_ui(self, row):
        name, color_str, cat = row[0], str(row[1]) if row[1] else "Colorless", row[3]
        display = self.clean_row(row)
        
        self.tree_dict["Master"]["All"].insert("", "end", values=display)
        
        dest_colors = []
        standard = ["White", "Blue", "Black", "Red", "Green"]
        detected = [c for c in standard if c in color_str]
        
        if len(detected) > 1:
            dest_colors.append("Multicolor")
            dest_colors.extend(detected)
        elif len(detected) == 1:
            dest_colors.append(detected[0])
        else:
            dest_colors.append("Colorless")

        for c_tab in dest_colors:
            if c_tab in self.tree_dict:
                self.tree_dict[c_tab]["All"].insert("", "end", values=display)
                if cat in self.tree_dict[c_tab]:
                    self.tree_dict[c_tab][cat].insert("", "end", values=display)

    def update_fields_visibility(self, event=None):
        for w in self.dynamic_frame.winfo_children(): w.grid_forget()
        ctype = self.var_type.get()
        r, c = 0, 0
        
        show = {"mana":True, "pt":False, "sub":False, "key":False, "ab":False, "loy":False}
        if ctype == "Land": show["mana"]=False; show["ab"]=True
        elif "Creature" in ctype or "Vehicle" in ctype: show["pt"]=True; show["sub"]=True; show["key"]=True
        elif ctype == "Planeswalker": show["loy"]=True
        if ctype in ["Legendary Creature", "Creature", "Artifact Creature"]: show["sub"]=True; show["key"]=True

        widgets = [("mana", self.lbl_mana, self.ent_mana), ("pt", self.lbl_pt, self.ent_pt), ("sub", self.lbl_sub, self.ent_sub),
                   ("key", self.lbl_key, self.ent_key), ("ab", self.lbl_ab, self.ent_ab), ("loy", self.lbl_loy, self.ent_loy)]
        
        for key, lbl, ent in widgets:
            if show[key]:
                lbl.grid(row=r, column=c, sticky="e"); ent.grid(row=r, column=c+1, sticky="w", padx=5)
                c += 2
                if c > 6: r+=1; c=0

    def clean_row(self, row):
        l = list(row)
        q = l.pop(2)
        l.insert(0, q)
        cleaned = []
        for x in l:
            if x is None or str(x).lower() == 'nan':
                cleaned.append("")
            else:
                cleaned.append(str(x).replace('\n', ' '))
        return cleaned

    def clear_form(self):
        self.editing_card_id = None
        self.btn_submit.config(text="Add Card", bg="#dddddd", command=self.add_card)
        self.btn_delete.config(state="disabled")
        for v in [self.var_name, self.var_color, self.var_type, self.var_mana_cost, self.var_pt, self.var_subtypes, self.var_keywords, self.var_ability, self.var_loyalty]: v.set("")
        self.var_quantity.set(1)
        self.update_fields_visibility()

    def process_excel_file(self, file_path):
        xls = pd.read_excel(file_path, sheet_name=None, header=None)
        cat_map = {"Not Basic Lands": "Land", "Legendary Creatures": "Legendary Creature", "Ledgendary Creatures": "Legendary Creature", "Creatures": "Creature", "Planeswalkers": "Planeswalker", "Instants": "Instant", "Sorceries": "Sorcery", "Legendary Enchantments - Backgrounds": "Legendary Enchantment", "Ledgendary Enchantments - Backgrounds": "Legendary Enchantment", "Legendary Enchantment Creature - Backgrounds": "Legendary Creature", "Ledgendary Enchantment Creature - Backgrounds": "Legendary Creature", "Enchantments": "Enchantment", "Legendary Artifact": "Legendary Artifact", "Ledgendary Artifact": "Legendary Artifact", "Ledgendary Artifacts": "Legendary Artifact", "Artifacts": "Artifact", "Artifact Equiment": "Artifact Equipment", "Artifact Vehicle": "Artifact Vehicle", "Artifact Treasure": "Artifact Treasure", "Atrifact Creatures": "Artifact Creature", "Articaft Cretures": "Artifact Creature"}

        for sheet_name, df in xls.items():
            if sheet_name in ["All", "Statistics"] or df.empty: continue
            j = 0
            while j < df.shape[1]:
                cat_raw = df.iloc[0, j]
                if pd.notna(cat_raw) and isinstance(cat_raw, str) and cat_raw != "Total Cards":
                    mapped_cat = cat_map.get(cat_raw.strip(), cat_raw.strip())
                    block_cols = []
                    k = j
                    while k < df.shape[1]:
                        header = df.iloc[1, k]
                        if pd.isna(header): break
                        block_cols.append(str(header).strip())
                        k += 1
                    if not block_cols: j += 1; continue

                    block_data = df.iloc[2:, j:k].copy()
                    block_data.columns = block_cols
                    if "Name" in block_data.columns:
                        block_data = block_data.dropna(subset=["Name"])
                        for _, row in block_data.iterrows():
                            c_val = row["Color"] if "Color" in row and pd.notna(row["Color"]) else sheet_name
                            data = {"name": row["Name"], "card_category": mapped_cat, "quantity": int(row["Quantity"]) if "Quantity" in row and pd.notna(row["Quantity"]) else 1, "color": c_val, "mana_cost": str(row["Cost"]) if "Cost" in row and pd.notna(row["Cost"]) else None, "power_toughness": str(row["Power/Toughness"]) if "Power/Toughness" in row and pd.notna(row["Power/Toughness"]) else None, "subtypes": str(row["Types"]) if "Types" in row and pd.notna(row["Types"]) else None, "keywords": str(row["Keywords"]) if "Keywords" in row and pd.notna(row["Keywords"]) else None, "ability": str(row["Ability"]) if "Ability" in row and pd.notna(row["Ability"]) else None, "loyalty": int(row["Base Loyalty"]) if "Base Loyalty" in row and pd.notna(row["Base Loyalty"]) else None}
                            for key, val in data.items():
                                if str(val) == 'nan': data[key] = None
                            self.upsert_card(data)
                    j = k
                else: j += 1

In [29]:
try:
    if 'root' in locals() and root:
        root.destroy()
except Exception:
    pass

root = TkinterDnD.Tk()
app = MTGCardManager(root)
root.mainloop()